# BC5CDR exploratory data analysis

Quick look at the dataset before training: label distribution, sentence length, and a few example sentences with their entity tags. This is the data the `TransformerNER` model and the BERT baseline are both trained and evaluated on — see `src/data/dataset.py` and `DECISIONS.md` for how it's loaded and aligned.

In [ ]:
import sys
sys.path.insert(0, "..")

from src.data.dataset import parse_conll, LABEL2ID, ID2LABEL

train = parse_conll("../data/raw/train.txt")
dev = parse_conll("../data/raw/dev.txt")
test = parse_conll("../data/raw/test.txt")

print(f"train: {len(train)} sentences")
print(f"dev:   {len(dev)} sentences")
print(f"test:  {len(test)} sentences")
print(f"\nLabel set: {LABEL2ID}")

## Sentence length distribution

In [ ]:
import matplotlib.pyplot as plt

lengths = [len(ex["tokens"]) for ex in train]

print(f"min length:    {min(lengths)}")
print(f"max length:    {max(lengths)}")
print(f"mean length:   {sum(lengths)/len(lengths):.1f}")
print(f"sentences over max_seq_len=128: {sum(1 for l in lengths if l > 128)} "
      f"({100*sum(1 for l in lengths if l > 128)/len(lengths):.2f}%)")

plt.figure(figsize=(8, 4))
plt.hist(lengths, bins=40)
plt.axvline(128, color="red", linestyle="--", label="max_seq_len=128")
plt.xlabel("sentence length (word tokens)")
plt.ylabel("count")
plt.title("BC5CDR train sentence length distribution")
plt.legend()
plt.tight_layout()
plt.show()

Only a tiny fraction of sentences exceed `max_seq_len=128`, so truncation (see `DECISIONS.md`) affects very few examples — confirms the sequence length choice in `config.yaml` is reasonable for this dataset.

## Label distribution

In [ ]:
from collections import Counter

all_tags = [tag for ex in train for tag in ex["ner_tags"]]
tag_counts = Counter(all_tags)

for tag_id, count in sorted(tag_counts.items()):
    label = ID2LABEL[tag_id]
    pct = 100 * count / len(all_tags)
    print(f"{label:12s} {count:7d}  ({pct:.2f}%)")

As expected for NER, `O` overwhelmingly dominates — this is exactly why entity-level F1 (not token accuracy) is the right evaluation metric, see `DECISIONS.md`. A model predicting `O` everywhere would score very high on accuracy while finding zero entities.

## Example sentences

In [ ]:
def show_example(ex, idx):
    print(f"--- sentence {idx} ---")
    for tok, tag in zip(ex["tokens"], ex["ner_tags"]):
        label = ID2LABEL[tag]
        marker = "  <-- entity" if label != "O" else ""
        print(f"  {tok:20s} {label}{marker}")
    print()

# A short example
show_example(train[0], 0)

In [ ]:
# An example with multiple entities, to see B-/I- boundaries in context
for i, ex in enumerate(train[:200]):
    entity_count = sum(1 for t in ex["ner_tags"] if t != LABEL2ID["O"])
    if entity_count >= 4:
        show_example(ex, i)
        break

## Takeaways

- ~3,942 training sentences, label set is `O` / `B-Entity` / `I-Entity` (IO-style BIO tagging — no separate chemical vs. disease entity types in this particular CoNLL mirror, see `DECISIONS.md` for the dataset-loading story).
- Sentence lengths are short enough that `max_seq_len=128` truncates almost nothing.
- `O` dominates the label distribution by a wide margin, which is the reason entity-level F1 (seqeval) is used for evaluation instead of token accuracy.